## Self-Corrective RAG - 검색 결과 자기평가·재시도

[02_Agentic RAG](02_Agentic%20RAG%20-%20Agent%EA%B0%80%20%EA%B2%80%EC%83%89%20%EC%A0%84%EB%9E%B5%20%EC%9E%90%EC%9C%A8%20%EA%B2%B0%EC%A0%95.ipynb) 노트북의 마지막 정리에서 남겨둔 한계가 있었다.

> 이 노트북은 도구 결과를 그대로 신뢰하지만, **Self-RAG/CRAG**처럼 검색 결과의 관련성을 LLM이 별도로
> 채점(grading)하고 낮으면 질의를 재작성(query rewriting)하는 단계를 추가하면 더 견고해진다.

Agentic RAG는 "어떤 도구를 쓸지"는 스스로 고르지만, 일단 도구가 뭔가를 반환하면 그 내용을 검증 없이
믿고 답을 만든다. 검색된 문서가 질문과 무관하거나 애매한 표현 때문에 엉뚱한 문서가 잡혀도 그대로
답변에 섞여 들어간다.

이 노트북은 **CRAG(Corrective RAG)** 방식을 LangGraph로 구현한다.

1. 벡터 검색으로 문서를 가져온다.
2. LLM이 검색된 문서 하나하나를 질문과 **관련 있는지(yes/no) 채점**한다.
3. 관련 문서가 하나도 없으면, 질문을 더 명확하게 **재작성**하고 다시 검색한다. (재시도 한도 있음)
4. 재시도를 다 써도 관련 문서를 못 찾으면, 사내 지식베이스 바깥의 **외부 지식(웹 검색)** 으로 전환한다.
5. 그래도 못 찾으면 "모른다"라고 답한다.

즉 "검색했다"와 "검색이 잘 됐다"를 구분하고, 후자가 아니면 스스로 교정하는 루프를 만드는 것이 핵심이다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
load_dotenv("C:/env/.env")

True

### [0] 공통 준비: LLM, 임베딩 모델

In [2]:
from typing import Annotated, List, TypedDict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_2036\1172240248.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### [1] 샘플 도메인 재구성: 테크노바 지식베이스

01·02번 노트북과 동일한 "테크노바" 조직·프로젝트 사실 문장을 **사내 지식베이스(내부 벡터 스토어)** 로 사용한다.

여기에 더해, 사내 지식베이스에는 없는 **회사 일반 정보**(설립 연도, 본사 위치, 대표이사, 투자 유치 이력)를
별도의 "웹 검색 결과" 대용 지식베이스로 준비한다. 실전에서는 이 부분이 Tavily 같은 실제 웹 검색 API
호출로 대체되며, 여기서는 노트북을 API 키 없이도 독립 실행할 수 있도록 검색 가능한 벡터 스토어로 흉내낸다.

In [4]:
company_facts = [
    "김민준은 테크노바의 AI팀 소속이다.",
    "이서연은 AI팀의 팀장이다.",
    "AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.",
    "박지훈은 데이터팀 소속이다.",
    "최유진은 데이터팀의 팀장이다.",
    "데이터팀은 '추천시스템 고도화' 프로젝트를 담당한다.",
    "AI팀은 데이터팀과 긴밀히 협업한다.",
    "정다은은 인프라팀 소속이다.",
    "한소희는 인프라팀의 팀장이다.",
    "인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.",
    "데이터팀은 인프라팀과 긴밀히 협업한다.",
    "'그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.",
    "박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.",
    "김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.",
    "프로덕트팀은 '온보딩 자동화' 프로젝트를 담당한다.",
    "정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.",
]

web_facts = [
    "테크노바는 2016년에 설립된 IT 기업이다.",
    "테크노바의 본사는 서울特별시 강남구에 위치한다.",
    "테크노바의 대표이사는 강태민이다.",
    "테크노바는 2023년에 시리즈B 투자를 유치했다.",
]


def to_docs(facts, source):
    return [
        Document(page_content=fact, metadata={"source": source, "fact_id": i})
        for i, fact in enumerate(facts)
    ]


internal_docs = to_docs(company_facts, "internal")
web_docs = to_docs(web_facts, "web")

vectorstore = FAISS.from_documents(internal_docs, embeddings)
web_vectorstore = FAISS.from_documents(web_docs, embeddings)


def format_docs(documents: List[Document]) -> str:
    return "\n".join(f"- {d.page_content}" for d in documents)


print(f"사내 지식베이스 문장 수: {len(company_facts)}")
print(f"외부(웹) 지식베이스 문장 수: {len(web_facts)}")

사내 지식베이스 문장 수: 16
외부(웹) 지식베이스 문장 수: 4


### [2] 검색 결과 채점기 (Retrieval Grader)

검색된 문서를 하나씩 LLM에게 보여주고 "이 문서가 질문에 답하는 데 실제로 쓸모 있는가"를 `yes`/`no`로
채점하게 한다. 유사도 점수(코사인 거리)가 아니라 **LLM이 내용을 읽고 내리는 판단**이라는 점이 핵심이다.
임베딩 유사도가 높아도 실제로는 질문과 무관한 문서가 섞여 들어오는 경우를 걸러낸다.

In [5]:
from pydantic import BaseModel, Field


class GradeDocument(BaseModel):
    binary_score: str = Field(
        description="문서가 질문에 답하는 데 관련이 있으면 'yes', 없으면 'no'"
    )


grade_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 검색된 문서가 사용자 질문과 관련이 있는지 채점하는 채점자다.\n"
            "문서에 질문과 관련된 키워드나 의미가 담겨 있으면 관련 있다고 판단한다.\n"
            "엄격한 정답 일치가 아니라, 답을 찾는 데 실제로 도움이 되는지를 기준으로 느슨하게 채점하되,\n"
            "명백히 무관한 주제라면 'no'로 채점하라.",
        ),
        ("human", "[검색된 문서]\n{document}\n\n[질문]\n{question}"),
    ]
)
doc_grader = llm.with_structured_output(GradeDocument)
grade_chain = grade_prompt | doc_grader

# 채점기 단독 테스트
test_score = grade_chain.invoke(
    {"document": "이서연은 AI팀의 팀장이다.", "question": "이서연은 어느 팀 팀장이야?"}
)
print(f"관련 문서 채점 예시: {test_score.binary_score}")

test_score2 = grade_chain.invoke(
    {"document": "박지훈은 데이터팀 소속이다.", "question": "이서연은 어느 팀 팀장이야?"}
)
print(f"무관 문서 채점 예시: {test_score2.binary_score}")

관련 문서 채점 예시: yes
무관 문서 채점 예시: no


### [3] 질의 재작성기 (Query Rewriter)

관련 문서를 하나도 못 찾았을 때, 원래 질문을 그대로 다시 검색해봤자 결과는 똑같다. 질문에 섞인
구어체 표현이나 모호한 지칭을 정리해 **벡터 검색에 더 적합한 질의**로 다시 쓰게 한다.

In [6]:
rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "다음 질문으로 검색했지만 관련 문서를 찾지 못했다. 검색에 더 적합하도록 질문을 다시 작성하라.\n"
            "- 원래 질문의 의도는 그대로 유지한다.\n"
            "- 구어체 표현이나 축약된 지칭을 명확한 핵심 키워드로 풀어 쓴다.\n"
            "- 재작성한 질문 한 줄만 출력하고, 다른 설명은 덧붙이지 않는다.",
        ),
        ("human", "원래 질문: {question}"),
    ]
)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

print(rewrite_chain.invoke({"question": "미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?"}))

미준이가 만드는 프로젝트는 누가 담당하나요?


### [4] 답변 생성 체인

관련 문서(사내 지식베이스 또는 웹 검색 결과)만 근거로 답한다. 근거 출처가 어디인지도 답변에 함께
밝히게 해서, 사용자가 "이 답이 사내 확정 사실인지 외부 지식인지"를 구분할 수 있게 한다.

In [7]:
generate_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [근거] 문서만 사용해 질문에 답하라. 문서에 없는 내용은 추측하지 말고 답하지 않는다.\n"
            "답변 끝에 근거 출처가 사내 지식베이스인지 외부(웹) 검색 결과인지 한 줄로 밝혀라.",
        ),
        ("human", "[근거 출처: {source}]\n{context}\n\n[질문]\n{question}"),
    ]
)
generate_chain = generate_prompt | llm | StrOutputParser()

### [5] LangGraph로 자기평가·재시도 루프 구성

흐름은 다음과 같다.

1. `retrieve`: 현재 질의로 사내 지식베이스를 벡터 검색한다.
2. `grade_documents`: 검색된 문서를 하나씩 채점해 관련 문서만 남긴다.
3. 관련 문서가 있으면 `generate`로, 없으면 재시도 횟수가 남아 있는 한 `transform_query` → `retrieve`로
   돌아간다(재검색 루프).
4. 재시도를 다 썼는데도 관련 문서가 없으면 `web_search`로 전환해 외부 지식에서 다시 찾는다.
5. `web_search` 이후에는(찾았든 못 찾았든) `generate`로 가서 최종 답을 만든다. 근거가 전혀 없으면
   "모른다"라고 답한다.

`retry_count`로 재검색 루프 횟수를 제한해, 관련 문서를 영영 못 찾는 질문에서도 무한 루프에 빠지지
않게 한다.

In [8]:
import operator

from langgraph.graph import StateGraph, END

MAX_RETRIES = 2


class CRAGState(TypedDict):
    question: str
    original_question: str
    documents: List[Document]
    retry_count: int
    used_web: bool
    generation: str
    trace: Annotated[List[str], operator.add]


def retrieve(state: CRAGState) -> dict:
    documents = vectorstore.similarity_search(state["question"], k=4)
    return {
        "documents": documents,
        "trace": [f"[검색] 질의 '{state['question']}' → 사내 KB에서 {len(documents)}건"],
    }


def grade_documents(state: CRAGState) -> dict:
    trace_lines = []
    relevant_docs = []
    for d in state["documents"]:
        score = grade_chain.invoke(
            {"document": d.page_content, "question": state["original_question"]}
        ).binary_score
        mark = "관련" if score == "yes" else "무관"
        trace_lines.append(f"    · [{mark}] {d.page_content}")
        if score == "yes":
            relevant_docs.append(d)
    trace_lines.insert(
        0, f"[채점] 검색된 {len(state['documents'])}건 중 관련 문서 {len(relevant_docs)}건"
    )
    return {"documents": relevant_docs, "trace": trace_lines}


def decide_to_generate(state: CRAGState) -> str:
    if state["documents"]:
        return "generate"
    if state["retry_count"] < MAX_RETRIES:
        return "transform_query"
    return "web_search"


def transform_query(state: CRAGState) -> dict:
    new_question = rewrite_chain.invoke({"question": state["question"]}).strip()
    next_retry = state["retry_count"] + 1
    return {
        "question": new_question,
        "retry_count": next_retry,
        "trace": [
            f"[질의 재작성] '{state['question']}' → '{new_question}' (재시도 {next_retry}/{MAX_RETRIES})"
        ],
    }


def web_search(state: CRAGState) -> dict:
    retrieved = web_vectorstore.similarity_search(state["original_question"], k=3)
    return {
        "documents": retrieved,
        "used_web": True,
        "trace": [
            "[웹 검색 폴백] 사내 지식베이스 재시도를 모두 소진, 외부(웹) 지식으로 전환",
            f"    · 웹 검색 결과 {len(retrieved)}건 확보",
        ],
    }


def generate(state: CRAGState) -> dict:
    documents = state["documents"]
    if not documents:
        return {
            "generation": "사내 지식베이스와 웹 검색을 모두 확인했지만 근거를 찾지 못했다. 모른다.",
            "trace": ["[생성] 근거 없음 → 모른다고 응답"],
        }
    source = "외부(웹) 검색 결과" if state.get("used_web") else "사내 지식베이스"
    answer = generate_chain.invoke(
        {
            "context": format_docs(documents),
            "question": state["original_question"],
            "source": source,
        }
    )
    return {
        "generation": answer,
        "trace": [f"[생성] {source} {len(documents)}건을 근거로 최종 답변 작성"],
    }


workflow = StateGraph(CRAGState)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("transform_query", transform_query)
workflow.add_node("web_search", web_search)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_to_generate,
    {"generate": "generate", "transform_query": "transform_query", "web_search": "web_search"},
)
workflow.add_edge("transform_query", "retrieve")
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

self_corrective_rag = workflow.compile()

print(self_corrective_rag.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	transform_query(transform_query)
	web_search(web_search)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade_documents -.-> generate;
	grade_documents -.-> transform_query;
	grade_documents -.-> web_search;
	retrieve --> grade_documents;
	transform_query --> retrieve;
	web_search --> generate;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### [6] 실행 도우미: 채점·재시도 트레이스 출력

각 단계에서 어떤 문서가 관련/무관으로 채점됐는지, 질의가 어떻게 재작성됐는지, 웹 검색으로 넘어갔는지를
순서대로 확인할 수 있게 한다.

In [9]:
def run_self_corrective_rag(question: str, verbose: bool = True) -> str:
    init_state: CRAGState = {
        "question": question,
        "original_question": question,
        "documents": [],
        "retry_count": 0,
        "used_web": False,
        "generation": "",
        "trace": [],
    }
    result = self_corrective_rag.invoke(init_state, config={"recursion_limit": 15})

    if verbose:
        print(f"질문: {question}\n")
        for line in result["trace"]:
            print(line)
        print(f"\n최종 답변:\n{result['generation']}")

    return result["generation"]

### [7] 실험 1 — 정상적인 질문

질문이 명확하고 관련 문서가 바로 잡히는 경우, 재시도나 웹 검색 없이 1회 검색·채점만으로 끝나는지
확인한다.

In [10]:
_ = run_self_corrective_rag("이서연은 어느 팀 팀장이야?")

질문: 이서연은 어느 팀 팀장이야?

[검색] 질의 '이서연은 어느 팀 팀장이야?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 1건
    · [관련] 이서연은 AI팀의 팀장이다.
    · [무관] 한소희는 인프라팀의 팀장이다.
    · [무관] 최유진은 데이터팀의 팀장이다.
    · [무관] 박지훈은 데이터팀 소속이다.
[생성] 사내 지식베이스 1건을 근거로 최종 답변 작성

최종 답변:
이서연은 AI팀의 팀장이다. 

근거 출처: 사내 지식베이스


### [8] 실험 2 — 구어체·모호한 표현이 섞인 질문

의미는 통하지만 벡터 검색이 헷갈릴 만한 축약된 표현("미준이", "기대는")을 일부러 사용한 질문이다.
Naive RAG라면 top-k 안에 담당자가 다른 여러 프로젝트 문서가 섞여 들어와 "알 수 없다"로 답하기 쉽다.
채점 단계가 무관한 후보들을 걸러내 관련 문서만으로 정확히 답하는지 확인한다.

In [11]:
_ = run_self_corrective_rag("미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?")

질문: 미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?

[검색] 질의 '미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 2건
    · [관련] 김민준은 '그래프 RAG 엔진' 프로젝트의 담당자이다.
    · [무관] 정다은은 '검색 인프라 개선' 프로젝트의 담당자이다.
    · [무관] 박지훈은 '추천시스템 고도화' 프로젝트의 담당자이다.
    · [관련] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
[생성] 사내 지식베이스 2건을 근거로 최종 답변 작성

최종 답변:
미준이가 만드는 '그래프 RAG 엔진' 프로젝트는 AI팀이 맡고 있습니다. 

근거 출처: 사내 지식베이스


### [9] 실험 3 — 사내 KB에는 없고 웹(외부 지식)에만 있는 질문

사내 지식베이스에는 프로젝트·조직 정보만 있고 회사 설립 연도 같은 일반 정보는 없다. 재시도를 다
소진한 뒤 웹 검색 폴백으로 전환해 답을 찾는지 확인한다.

In [12]:
_ = run_self_corrective_rag("테크노바는 언제 설립됐어?")

질문: 테크노바는 언제 설립됐어?

[검색] 질의 '테크노바는 언제 설립됐어?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
    · [무관] '그래프 RAG 엔진' 프로젝트는 '검색 인프라 개선' 프로젝트의 결과물에 의존한다.
    · [무관] 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
[질의 재작성] '테크노바는 언제 설립됐어?' → '테크노바의 설립 연도는 언제인가요?' (재시도 1/2)
[검색] 질의 '테크노바의 설립 연도는 언제인가요?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
    · [무관] 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
    · [무관] 데이터팀은 인프라팀과 긴밀히 협업한다.
[질의 재작성] '테크노바의 설립 연도는 언제인가요?' → '테크노바의 설립 연도는 언제입니까?' (재시도 2/2)
[검색] 질의 '테크노바의 설립 연도는 언제입니까?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
    · [무관] 데이터팀은 인프라팀과 긴밀히 협업한다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
[웹 검색 폴백] 사내 지식베이스 재시도를 모두 소진, 외부(웹) 지식으로 전환
    · 웹 검색 결과 3건 확보
[생성] 외부(웹) 검색 결과 3건을 근거로 최종 답변 작성

최종 답변:
테크노바는 2016년에 설립되었습니다.  
근거 출처: 외부(웹) 검색 결과


### [10] 실험 4 — 사내 KB에도 웹에도 없는 질문

재시도와 웹 검색 폴백을 모두 거쳐도 근거를 찾지 못하면, 억지로 답을 지어내지 않고 "모른다"라고
답하는지 확인한다.

In [13]:
_ = run_self_corrective_rag("테크노바 AI팀의 이번 분기 매출 목표는 얼마야?")

질문: 테크노바 AI팀의 이번 분기 매출 목표는 얼마야?

[검색] 질의 '테크노바 AI팀의 이번 분기 매출 목표는 얼마야?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] AI팀은 데이터팀과 긴밀히 협업한다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
    · [무관] 데이터팀은 인프라팀과 긴밀히 협업한다.
[질의 재작성] '테크노바 AI팀의 이번 분기 매출 목표는 얼마야?' → '테크노바 인공지능 팀의 이번 분기 매출 목표는 얼마입니까?' (재시도 1/2)
[검색] 질의 '테크노바 인공지능 팀의 이번 분기 매출 목표는 얼마입니까?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] AI팀은 데이터팀과 긴밀히 협업한다.
    · [무관] 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
[질의 재작성] '테크노바 인공지능 팀의 이번 분기 매출 목표는 얼마입니까?' → '테크노바 인공지능 팀의 이번 분기 매출 목표는 얼마입니까?' (재시도 2/2)
[검색] 질의 '테크노바 인공지능 팀의 이번 분기 매출 목표는 얼마입니까?' → 사내 KB에서 4건
[채점] 검색된 4건 중 관련 문서 0건
    · [무관] 김민준은 테크노바의 AI팀 소속이다.
    · [무관] AI팀은 데이터팀과 긴밀히 협업한다.
    · [무관] 인프라팀은 '검색 인프라 개선' 프로젝트를 담당한다.
    · [무관] AI팀은 '그래프 RAG 엔진' 프로젝트를 담당한다.
[웹 검색 폴백] 사내 지식베이스 재시도를 모두 소진, 외부(웹) 지식으로 전환
    · 웹 검색 결과 3건 확보
[생성] 외부(웹) 검색 결과 3건을 근거로 최종 답변 작성

최종 답변:
해당 정보는 문서에 포함되어 있지 않으

### [11] Naive RAG vs Self-Corrective RAG 비교

같은 질문들을 (a) 채점·재시도 없이 top-k를 그대로 믿는 Naive RAG와 (b) 이 노트북의
Self-Corrective RAG에 각각 던져, 검색이 부실할 때 결과가 어떻게 갈리는지 비교한다.

In [14]:
naive_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 [사실] 목록만 근거로 질문에 답하라. 목록에 없는 내용은 추측하지 말고 "
            "'주어진 사실만으로는 알 수 없다'라고 답하라.\n\n[사실]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
naive_chain = naive_prompt | llm | StrOutputParser()


def naive_rag(question: str) -> str:
    retrieved = vectorstore.similarity_search(question, k=4)
    return naive_chain.invoke({"context": format_docs(retrieved), "question": question})


comparison_queries = [
    "이서연은 어느 팀 팀장이야?",
    "미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?",
    "테크노바는 언제 설립됐어?",
    "테크노바 AI팀의 이번 분기 매출 목표는 얼마야?",
]

for q in comparison_queries:
    naive = naive_rag(q)
    corrective = run_self_corrective_rag(q, verbose=False)
    print(f"질문: {q}")
    print(f"- Naive RAG (채점·재시도 없음)      : {naive}")
    print(f"- Self-Corrective RAG (채점·재시도) : {corrective}")
    print("-" * 80)

질문: 이서연은 어느 팀 팀장이야?
- Naive RAG (채점·재시도 없음)      : 이서연은 AI팀의 팀장이다.
- Self-Corrective RAG (채점·재시도) : 이서연은 AI팀의 팀장이다.  
근거 출처: 사내 지식베이스
--------------------------------------------------------------------------------
질문: 미준이가 만드는 거가 기대는 프로젝트는 누가 맡음?
- Naive RAG (채점·재시도 없음)      : 주어진 사실만으로는 알 수 없다.
- Self-Corrective RAG (채점·재시도) : 미준이가 만드는 '그래프 RAG 엔진' 프로젝트는 AI팀이 맡고 있습니다. 

근거 출처: 사내 지식베이스
--------------------------------------------------------------------------------
질문: 테크노바는 언제 설립됐어?
- Naive RAG (채점·재시도 없음)      : 주어진 사실만으로는 알 수 없다.
- Self-Corrective RAG (채점·재시도) : 테크노바는 2016년에 설립되었습니다.  
근거 출처: 외부(웹) 검색 결과
--------------------------------------------------------------------------------
질문: 테크노바 AI팀의 이번 분기 매출 목표는 얼마야?
- Naive RAG (채점·재시도 없음)      : 주어진 사실만으로는 알 수 없다.
- Self-Corrective RAG (채점·재시도) : 해당 정보는 문서에 포함되어 있지 않으므로 답변할 수 없습니다.  
근거 출처: 외부(웹) 검색 결과
--------------------------------------------------------------------------------


### [12] 정리

| 구분 | Naive RAG | Self-Corrective RAG |
|---|---|---|
| 검색 결과 신뢰 방식 | top-k를 그대로 신뢰 | LLM이 문서별로 관련성 채점 후 무관 문서 제외 |
| 검색이 부실할 때 대응 | 없음 (엉뚱한 문서로 답하거나 "알 수 없다") | 질의 재작성 후 재검색 (최대 `MAX_RETRIES`회) |
| 사내 지식에 없는 질문 | 답 불가 | 외부(웹) 지식으로 폴백 시도 |
| 그래도 못 찾을 때 | 모호하게 답하거나 "알 수 없다" | 근거 없음을 명시하고 "모른다"로 답 |
| 비용·지연 | 낮음 (검색 1회 + 생성 1회) | 높음 (문서별 채점 호출 + 재검색·웹검색 반복) |

**구현 포인트**
- 관련성 판단을 유사도 점수가 아니라 **LLM 채점(structured output)** 으로 분리해, 임베딩이 놓치는
  "의미상 무관함"을 걸러냈다.
- 관련 문서가 하나도 없을 때만 재작성·재검색을 트리거해서, 이미 잘 되는 검색까지 불필요하게
  반복하지 않도록 했다.
- `retry_count`로 재검색 루프 상한을 두어 계속 실패하는 질문에서도 비용이 무한정 늘지 않게 했다.
- 사내 지식베이스와 외부(웹) 지식을 **서로 다른 벡터 스토어**로 분리해, 웹 폴백으로 넘어갔는지
  여부와 답변의 근거 출처를 명확히 구분했다.

**한계와 실전 확장 포인트**
- 문서마다 별도 LLM 호출로 채점하므로 문서 수가 많아지면 채점 비용이 커진다. 배치 채점이나
  경량 분류 모델로 대체하면 비용을 줄일 수 있다.
- 여기서는 웹 검색을 오프라인 벡터 스토어로 흉내냈다. 실전에서는 Tavily, Bing 등 실제 검색
  API로 `web_search` 노드를 교체하면 된다.
- 웹 검색 결과도 채점 없이 바로 신뢰하는 단순화를 했다. 웹 결과 역시 `grade_documents`처럼
  관련성 채점을 거치게 하면 더 견고해진다.
- [02_Agentic RAG](02_Agentic%20RAG%20-%20Agent%EA%B0%80%20%EA%B2%80%EC%83%89%20%EC%A0%84%EB%9E%B5%20%EC%9E%90%EC%9C%A8%20%EA%B2%B0%EC%A0%95.ipynb)의
  자율 도구 선택과 이 노트북의 채점·재시도 루프를 결합하면, "어떤 도구를 쓸지"와 "그 결과가
  믿을 만한지"를 모두 스스로 판단하는 에이전트를 만들 수 있다.